In [ ]:
 # 教師あり学習の二値分類モデル及びTensorflow・Kerasを用いて問題解決する
 # 正解率(accuracy 50%以上)

In [42]:
# 必要なライブラリのimport
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Tensorflow
import tensorflow as tf

# データの分割
from sklearn.model_selection import train_test_split

# DataFrameで全ての列を表示する設定
pd.options.display.max_columns = None

In [43]:
dataset = pd.read_csv("./Amazon_Reviews.csv")
dataset.head()

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024"
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,"September 16, 2024"
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024"
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024"
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024"


In [44]:
dataset_dev = dataset[['Rating', 'Review Text']]
dataset_dev.head()

,Rating,Review Text
0,Rated 1 out of 5 stars,"I registered on the website, tried to order a ..."
1,Rated 1 out of 5 stars,Had multiple orders one turned up and driver h...
2,Rated 1 out of 5 stars,I informed these reprobates that I WOULD NOT B...
3,Rated 1 out of 5 stars,I have bought from Amazon before and no proble...
4,Rated 1 out of 5 stars,If I could give a lower rate I would! I cancel...


In [48]:
# 明示的にコピー作る（ワーニング出るので）
dataset_dev = dataset_dev.copy()
#　列名Ratingから、点数のみを抽出
import re
def extract_rating(text):
  match = re.search(r'Rated (\d+) out of 5 stars', str(text))
  return int(match.group(1)) if match else None

dataset_dev['Rating_score'] = dataset_dev['Rating'].apply(extract_rating) 

In [50]:
dataset_dev['Rating_score'].isnull().sum()

np.int64(159)

In [51]:
#　データ型の確認
dataset_dev['Rating_score'].dtype
# 欠損の数を確認
dataset_dev['Rating_score'].isnull().sum()
# 欠損を削除
dataset_dev = dataset_dev.dropna(subset=['Rating_score'])
# ラベル作成、スコア3は除外
dataset_dev['label'] = dataset_dev['Rating_score'].apply(lambda x: 1 if x >= 4 else (0 if x<= 2 else None))
dataset_dev = dataset_dev.dropna(subset=['label'])
# labelが4,5はポジティブ（=1）、1,2はネガティブ（=0）とする
dataset_dev['label'] = dataset_dev['label'].astype(int)
# ポジティブとネガティブのバランス確認（ネガティブ多め）
dataset_dev['label'].value_counts()

label
0    14350
1     5820
Name: count, dtype: int64

In [52]:
# 文字列データの前処理
def clean_text(text):
    # 小文字にする
    text = str(text).lower()
    # HTMLタグ削除
    text = re.sub(r'<.*?>', '', text)
    # 記号と数字削除
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 余分な空白削除
    text = re.sub(r'\s+', ' ', text).strip()
    return text

dataset_dev['Review_comment'] = dataset_dev['Review Text'].apply(clean_text)

dataset_prod = dataset_dev[['Rating_score', 'label', 'Review_comment']]
dataset_prod.head()

,Rating_score,label,Review_comment
0,1.0,0,i registered on the website tried to order a l...
1,1.0,0,had multiple orders one turned up and driver h...
2,1.0,0,i informed these reprobates that i would not b...
3,1.0,0,i have bought from amazon before and no proble...
4,1.0,0,if i could give a lower rate i would i cancell...


In [53]:
# 自然言語を数値に変換（意味付けなし）
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(dataset_prod['Review_comment'])

X = tokenizer.texts_to_sequences(dataset_prod['Review_comment'])

from tensorflow.keras.preprocessing.sequence import pad_sequences

X = pad_sequences(X, maxlen=100)

y = dataset_prod['label']

In [54]:
# 数値にした単語のベクトル変換（意味付け）
# 文脈は考慮していない
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

#モデル定義
model = Sequential([
    Embedding(input_dim=10000, output_dim=16),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [55]:
# モデルの学習の定義
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# モデルの学習
model.fit(
    X, y,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

y_pred = model.predict(X)
y_pred_label = (y_pred > 0.5).astype(int)

from sklearn.metrics import confusion_matrix

confusion_matrix(y, y_pred_label)

from sklearn.metrics import classification_report

print(classification_report(y, y_pred_label))

Epoch 1/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8459 - loss: 0.3832 - val_accuracy: 0.2762 - val_loss: 0.9900
Epoch 2/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8818 - loss: 0.2821 - val_accuracy: 0.8017 - val_loss: 0.5284
Epoch 3/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9256 - loss: 0.1903 - val_accuracy: 0.8699 - val_loss: 0.3679
Epoch 4/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9441 - loss: 0.1467 - val_accuracy: 0.9113 - val_loss: 0.2697
Epoch 5/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9530 - loss: 0.1231 - val_accuracy: 0.8934 - val_loss: 0.2960
Epoch 6/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9559 - loss: 0.1131 - val_accuracy: 0.7702 - val_loss: 0.4638
Epoch 7/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9608 - loss: 0.1001 - val_accuracy: 0.7955 - val_loss: 0.4184
Epoch 8/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9632 - loss: 0.0926 - val_accuracy: 0.